# Experiment 2: how do vocal effects affect source separation?

Loudness-matched reverb, delay, compression and bitcrushing (4 levels each) applied to the vocal stem of the 50 MUSDB18 test songs, separated with Demucs and Spleeter.

This notebook is a thin front end: every step calls a script in `src/`, so the same commands also work in a terminal. **Every step skips work that is already done**, so it is safe to stop and rerun. The finished results are already in `results/exp2/`, so you can jump straight to section 5.

Set your paths in `src/config.py` (or the `SSS_DATA_ROOT` and `MUSDB18_ROOT` environment variables) first, and add the impulse response described in `assets/README.md`.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Interpreters for the three environments (see requirements/). Edit the last two.
PY_DATA = sys.executable                                   # this notebook's kernel: data-and-evaluation env
PY_SPLEETER = "/path/to/spleeter_env/bin/python"
PY_DEMUCS = "/path/to/demucs_env/bin/python"

def run(cmd):
    """Run a command and stream its output into the notebook."""
    with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")

## 1. Generate the loudness-matched mixes and reference stems
About 15 minutes for all 50 songs with 8 workers. Add `"--num", "1"` for a quick test.

In [ ]:
run([PY_DATA, "src/make_data_exp2.py", "--workers", "8"])

## 2. Separate with Demucs (about 1 to 2 hours; uses the GPU where available)
Add `"--limit", "3"` for a quick test.

In [ ]:
run([PY_DEMUCS, "-u", "src/run_demucs_exp2.py"])

## 3. Separate with Spleeter (about 1 hour; CPU, fresh Separator per file)
Do not raise `--workers` much above 3 without watching memory.

In [ ]:
run([PY_SPLEETER, "-u", "src/run_spleeter_exp2.py", "--workers", "3"])

## 4. Check that every output exists and has the right length, then score
SDR against the exact stems in each mix, ΔSDR against the no-effect condition, and the dry-reference analysis (about 20 minutes for 50 songs).

In [ ]:
sys.path.insert(0, str(REPO / "src"))
import soundfile as sf
from config import MIX_DIR, MODELS, STEMS

for name, root in MODELS.items():
    missing = wrong = 0
    for mix in sorted(MIX_DIR.glob("*/*.wav")):
        n = sf.info(mix).frames
        for s in STEMS:
            p = root / mix.parent.name / mix.stem / f"{s}.wav"
            if not p.exists():
                missing += 1
            elif sf.info(p).frames != n:
                wrong += 1
    print(f"{name}: {missing} missing stems, {wrong} with the wrong length")

In [ ]:
run([PY_DATA, "src/evaluate_exp2.py", "--workers", "6", "--dry-reference"])

## 5. Figures, tables and results

In [ ]:
run([PY_DATA, "src/make_figures_exp2.py"])

In [ ]:
import pandas as pd
from IPython.display import Image, display

res = REPO / "results" / "exp2"
t = pd.read_csv(res / "tables" / "vocals_delta_sdr_summary.csv")
print("Median vocal delta-SDR (dB) vs. no effect:")
display(t.pivot_table(index=["effect", "level"], columns="model", values="median_dSDR").round(2))
for f in ["fig1_vocal_delta_sdr_by_level", "fig2_stems_by_effect_level4", "fig3_processed_vs_dry_reference_level4"]:
    display(Image(str(res / "figures" / f"{f}.png")))

## Reading the results
- Reverb and delay degrade the vocal estimate in a graded way; compression and bitcrushing cost under about 0.8 dB.
- The other stems barely change.
- Both models keep the effect in the vocal stem (they score much worse against the dry vocal).

Full methods and numbers: `docs/EXPERIMENT2_HANDOFF.md`.